In [1]:
import requests 
from bs4 import BeautifulSoup as bs
import pandas as pd
import tldextract
import numpy as np
import matplotlib.pylab as plt
import os
import re

In [2]:
def on_sale_chk(text):
    if len(text)<1:
        return False
    return 'domain' in text and 'sale' in text

def on_parked_chk(text):
    if len(text)<1:
        return True
    return 'domain' in text and 'park' in text

def on_Parked(text):
    if len(text)<1:
        return True
    return (('website' in text or 'content' in text) and 'unavailable' in text) or ('will' in text and 'soon' in text)

In [3]:
#returns html contents, textual character length, website size, status code, parked or on sale

def soupFromUrl(scrapeUrl):
    headers = {'User-Agent': 'Mozilla/5.0 (Windows; U; Windows NT 6.1; zh-CN) AppleWebKit/533+ (KHTML, like Gecko)'}
    try:
        req = requests.get(scrapeUrl, headers=headers, timeout=5)
        # print(req.status_code)
        req.close()
        if req.status_code == 200:
            # print(bs(req.text, 'html.parser').get_text().strip().replace('\n',' '))
            soup = bs(req.text,'lxml')

            # print(soup)

            text = ''

            if soup.body:
                text = re.sub(r'[^\w]', ' ',soup.body.get_text(' ', strip=True).lower())

            # print(soup)

            # print('text',text)
            # print(bs(req.text, 'html.parser'))
            # return [bs(req.text, 'html.parser'),len(req.text), len(req.content), req.status_code]
            # print([len(req.content), len(text), req.status_code, 0+(on_sale_chk(text) or on_Parked(text) or on_parked_chk(text))])
            return [len(req.content), len(text), req.status_code, 0+(on_sale_chk(text) or on_Parked(text) or on_parked_chk(text))]
        else:
            # return [-1,0,0,req.status_code]
            return [0,0,req.status_code,0]
    except:
        # return [-1,0,0,-1]
        return [0,0,-1,0]

In [4]:
headers = {'User-Agent': 'Mozilla/5.0 (Windows; U; Windows NT 6.1; zh-CN) AppleWebKit/533+ (KHTML, like Gecko)'}
req = requests.get('https://www.lycos.com/', headers=headers, timeout=5)
print(req.status_code)
req.close()
if req.status_code == 200:
    # print(bs(req.text, 'html.parser').get_text().strip().replace('\n',' '))
    soup = bs(req.text,'lxml')

    # print(soup)
    text = ''

    if soup.body:
        text = re.sub(r'[^\w]', ' ',soup.body.get_text(' ', strip=True).lower())

    print(soup)
    print('text',text)
    print([len(req.content), len(text), req.status_code, 0+(on_sale_chk(text) or on_Parked(text) or on_parked_chk(text))])

200
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8"/>
<meta content="IE=edge" http-equiv="X-UA-Compatible"/>
<meta content="width=device-width, initial-scale=1" name="viewport"/>
<!-- The above 3 meta tags *must* come first in the head; any other head content must come *after* these tags -->
<meta content="Lycos, Inc., is a web search engine and web portal established in 1994, spun out of Carnegie Mellon University. Lycos also encompasses a network of email, webhosting, social networking, and entertainment websites." name="description"/>
<meta content="" name="author"/>
<link href="https://ly.lygo.net/static/lycos/img/favicon.ico" rel="icon" type="image/png"/>
<title>Lycos.com</title>
<link href="//fonts.googleapis.com/css?family=Lato:400,300,300italic,400italic,700,700italic" rel="stylesheet" type="text/css"/>
<link href="/css/in/fonts.css" rel="stylesheet" type="text/css"/>
<link href="https://ly.lygo.net/static/lycos/css/in/font-awesome.css" rel="stylesheet" type="text

In [5]:
# print(soupFromUrl('https://www.delinian.com/'))
# print(soupFromUrl('https://www.makecashonline.com/'))
print(soupFromUrl('https://www.lycos.com/'))

[13607, 328, 200, 0]


In [6]:
Korean_url = list(pd.read_csv('../Dataset/URL Data/Korean Dataset.csv',delimiter='\t')['URL'])
Korean_url[:10]

['www.coinonve.com',
 'https://play.google.com/store/apps/details?id=com.teamviewer.quicksuport.market엄마이링크클릭해서설치하고열기하면귀하의아이디라고9자리로',
 'https://play.google.com',
 'https://ya.mba/3PZ',
 'https://url.kr/fn8ocq',
 'https://play.google.com/store/ap',
 'www.coinoena.com',
 'www.coincneo.com',
 'https://ko.gl/ac60',
 'https://utka.su/jpmh8']

In [7]:
import re

def find_first_slash_preceded_by_number(s):
    # Regular expression to find the first instance of a number followed by '/'
    match = re.search(r'\d+/', s)
    
    if match:
        return match.start() + len(match.group()) - 1  # Return the index of '/'
    else:
        return -1  # Return -1 if no match is found

# Example usage
string = "example77/test 88/test2 99/test3"
index = find_first_slash_preceded_by_number(string)
print(index)  # Outputs the index of the first '/' preceded by a number

9


In [8]:
unique_Korean_url = set(Korean_url )

In [9]:
for idx,i in enumerate(unique_Korean_url):
    if '..' in i:
        if 'www' in i:
            unique_Korean_url[idx] = ''
        else:
            unique_Korean_url[idx] = unique_Korean_url[idx].replace('..','.')

In [10]:
unique_Korean_url = [i for i in unique_Korean_url if i!='']

In [11]:
Korean_dataset = {'ham':list(pd.read_csv('../Dataset/URL Data/Korean Dataset Ham.csv',delimiter='\t')['URL']),'spam':list(pd.read_csv('../Dataset/URL Data/Korean Dataset Spam.csv',delimiter='\t')['URL'])}

In [12]:
Korean_url_in_ham = [0]*len(unique_Korean_url)
Korean_url_in_spam = [0]*len(unique_Korean_url)

for idx,i in enumerate(unique_Korean_url):
    for j in Korean_dataset['ham']:
        if not isinstance(j, str):
            continue
        if i in j:
            Korean_url_in_ham[idx] = 1
            break
    for j in Korean_dataset['spam']:
        if not isinstance(j, str):
            continue
        if i in j:
            Korean_url_in_spam[idx] = 1
            break

print(Korean_url_in_ham.count(1))
print(Korean_url_in_spam.count(1))

3
116


In [13]:
common_urls = []
for i in range(len(Korean_url_in_ham)):
    if Korean_url_in_ham[i]==Korean_url_in_spam[i]:
        common_urls.append(unique_Korean_url[i])

len(common_urls)

1

In [14]:
Korean_dataset['Unique Url'] = unique_Korean_url

In [15]:
import tldextract

def FQDN(Url):
    
    url_extract_res = tldextract.extract(Url)
    fqdn = ''
    if url_extract_res.subdomain:
        fqdn = url_extract_res.subdomain + '.' + url_extract_res.domain + '.' + url_extract_res.suffix
        # fqdn = url_extract_res.domain + '.' + url_extract_res.suffix
    else:
        fqdn = url_extract_res.domain + '.' + url_extract_res.suffix
    
    return fqdn

In [16]:
Korean_dataset['FQDN'] = [FQDN(i) for i in Korean_dataset['Unique Url']]

len(set(Korean_dataset['FQDN']))

unable to cache publicsuffix.org-tlds.{'urls': ('https://publicsuffix.org/list/public_suffix_list.dat', 'https://raw.githubusercontent.com/publicsuffix/list/master/public_suffix_list.dat'), 'fallback_to_snapshot': True} in c:\ProgramData\anaconda3\Lib\site-packages\tldextract\.suffix_cache/publicsuffix.org-tlds\de84b5ca2167d4c83e38fb162f2e8738.tldextract.json. This could refresh the Public Suffix List over HTTP every app startup. Construct your `TLDExtract` with a writable `cache_dir` or set `cache_dir=False` to silence this warning. [WinError 5] Access is denied: 'c:\\ProgramData\\anaconda3\\Lib\\site-packages\\tldextract\\.suffix_cache'


75

In [17]:
a = soupFromUrl('https://facebook.com')

print(a)

[74355, 645, 200, 0]


The below query last ran on 7 Novemebr 2024

In [18]:
website_size, text_content_length, status_code, parked = [],[],[],[]

for i in Korean_dataset['FQDN']:
    a = soupFromUrl('https://'+i)

    website_size.append(a[0])
    text_content_length.append(a[1])
    status_code.append(a[2])
    parked.append(a[3])


# parked = [0]*len(Korean_dataset)

# for idx,i in enumerate(Korean_dataset['FQDN']):
#     if Korean_dataset['Status Code'][idx]==200:
#         a = soupFromUrl('https://'+i)
#         parked[idx] = a[3]
#         # break

# print(parked)

In [19]:
Korean_dataset['Website Size in KB'] = website_size
Korean_dataset['Website Textual Content Length'] = text_content_length
Korean_dataset['Status Code'] = status_code

Korean_dataset['Parked'] = parked

In [20]:
for i in Korean_dataset:
    print(len(Korean_dataset[i]))

3
127
120
120
120
120
120
120


In [21]:
Korean_dataset['ham'] = Korean_url_in_ham
Korean_dataset['spam'] = Korean_url_in_spam

In [22]:
Korean_dataset

{'ham': [0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  1,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  1,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  1,
  0,
  0,
  0,
  0],
 'spam': [1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,

In [23]:
# Korean_dataset = pd.read_csv('../Dataset/URL Data/Korean Websites Analysis.csv')
Korean_dataset = pd.DataFrame.from_dict(Korean_dataset)
Korean_dataset.head()

,ham,spam,Unique Url,FQDN,Website Size in KB,Website Textual Content Length,Status Code,Parked
0,0,1,www.coincneo.com,www.coincneo.com,0,0,-1,0
1,0,1,http://tinyurl.com/yfuwrq28,tinyurl.com,13574,0,200,1
2,0,1,http://tinyurl.com/y7yxy5gm,tinyurl.com,13574,0,200,1
3,0,1,http://fff.kr/auEM,fff.kr,0,0,-1,0
4,0,1,https://soo.gd/u053?sco,soo.gd,0,0,-1,0


In [24]:
Korean_dataset['Status Code'].value_counts()

Status Code
 200    64
-1      49
 404     3
 403     3
 400     1
Name: count, dtype: int64

In [25]:
numbers_to_replace = [501,403, 401]

# Value to replace with
new_value = 200

# Update the column
Korean_dataset.loc[Korean_dataset['Status Code'].isin(numbers_to_replace), 'Status Code'] = new_value

In [26]:
Korean_dataset['Status Code'].value_counts()

Status Code
 200    67
-1      49
 404     3
 400     1
Name: count, dtype: int64

In [33]:
print(len(Korean_dataset[(Korean_dataset['Status Code']==200) & (Korean_dataset['ham']==1)]))
print(len(Korean_dataset[(Korean_dataset['Status Code']==200) & (Korean_dataset['ham']==0)]))

3
64


In [28]:
Korean_dataset['Parked'].value_counts()

Parked
0    96
1    24
Name: count, dtype: int64

In [29]:
print(len(Korean_dataset[(Korean_dataset['Parked']==1) & (Korean_dataset['ham']==1)]))
print(len(Korean_dataset[(Korean_dataset['Parked']==1) & (Korean_dataset['ham']==0)]))

2
22


In [30]:
Korean_dataset.head()

,ham,spam,Unique Url,FQDN,Website Size in KB,Website Textual Content Length,Status Code,Parked
0,0,1,www.coincneo.com,www.coincneo.com,0,0,-1,0
1,0,1,http://tinyurl.com/yfuwrq28,tinyurl.com,13574,0,200,1
2,0,1,http://tinyurl.com/y7yxy5gm,tinyurl.com,13574,0,200,1
3,0,1,http://fff.kr/auEM,fff.kr,0,0,-1,0
4,0,1,https://soo.gd/u053?sco,soo.gd,0,0,-1,0


In [31]:
Korean_dataset.to_csv('../Dataset/URL Data/Korean Websites Analysis.csv', index=None)